# smICA callback-backed automation commands

Scientific actions are exposed under stable public names. Every command in this notebook runs on the Dear PyGui thread, writes only documented controls and invokes the callback already used by the graphical interface. The command layer does not contain a second PTU, ROI, filtering or concentration implementation.

## Interaction model

A client sends a public command such as `phot2conc.set_calibration`. The adapter validates its scientific arguments, writes the corresponding calibration controls, invokes `callback_omega_input`, `callback_kappa_input`, and the existing brightness callbacks, then returns normalized state. Widget tags and Python callback names remain private implementation details.

Operation order remains under client control. Select the mode first, load a directory, set controls, and then request calculation or export. `OPERATION_UNAVAILABLE` means that the requested mode or prerequisite GUI state is missing. `INVALID_PARAMETERS` identifies malformed public arguments.

## Commands

| Family | Commands |
|---|---|
| System and session | `system.health`, `system.version`, `system.capabilities`, `system.shutdown`, `session.inspect` |
| Mode | `gui.select_mode` |
| Extraction input and state | `extract.load_directory`, `extract.select_file`, `extract.get_state` |
| Extraction controls | `extract.set_gate`, `extract.set_rows_to_skip`, `extract.set_statistical_filter` |
| Statistical filters | `extract.open_filter_setup`, `extract.set_background_range`, `extract.open_decay_library`, `extract.select_library_decay`, `extract.import_library_decays`, `extract.calculate_filters`, `extract.accept_filters`, `extract.decline_filters`, `extract.select_filter` |
| Extraction execution | `extract.apply_current`, `extract.apply_all` |
| Phot2conc input and state | `phot2conc.load_directory`, `phot2conc.load_roi_directory`, `phot2conc.select_file`, `phot2conc.get_state` |
| Display and lifetime | `phot2conc.set_display`, `phot2conc.set_decay_range` |
| Calibration and errors | `phot2conc.load_calibration`, `phot2conc.set_calibration`, `phot2conc.set_error_mode` |
| ROI | `phot2conc.set_roi_source`, `phot2conc.set_auto_roi`, `phot2conc.select_roi` |
| Calculation | `phot2conc.calculate`, `phot2conc.add_result`, `phot2conc.calculate_all` |
| Export | `phot2conc.set_export_options`, `phot2conc.export_results` |

`system.capabilities` is the authoritative list exposed by the running version. `system.shutdown` is registered only for controlled test processes started with `--automation-allow-shutdown`; normal application instances reject remote shutdown.

> **Documentation added for v2.2.0**
>

The library/filter commands use explicit parameters: `extract.set_background_range` requires `channel` and `enabled`; `extract.open_decay_library` and `extract.import_library_decays` require `channel`; `extract.select_library_decay` and `extract.select_filter` require `channel` and the exact displayed `name`. Channels accept `1` or `2`. Missing prerequisites return `OPERATION_UNAVAILABLE` together with available decay or filter names where applicable.

## Example command sequence

```bash
smica-call gui.select_mode '{"mode":"phot2conc"}'
smica-call phot2conc.load_directory '{"directory":"/absolute/data/path"}'
smica-call phot2conc.load_calibration '{"path":"/absolute/path/to/calib.json"}'
smica-call phot2conc.set_roi_source '{"source":"auto"}'
smica-call phot2conc.set_auto_roi '{"channel":1,"cell_threshold":0.85,"many_cells":false}'
smica-call phot2conc.calculate '{}'
smica-call phot2conc.add_result '{}'
smica-call phot2conc.export_results '{"path":"/absolute/output/results.csv"}'
```

The client can send the sequence one command at a time or from a shell/Python workflow. Article workflows use the same public commands, preserve resume state and finish with a concise processing report.

## Statistical-filter sequence

Statistical-filter automation follows the existing GUI flow. Enable filtering, initialize the setup for a channel and either enable automatic background selection or import a compatible decay from the library. Calculate and accept the filter, select the accepted foreground filter, and then execute the native batch callback. The automation handlers pass the real sender tags and callback arguments used by the corresponding controls.


For a saved decay pattern, use this order:

1. `extract.set_statistical_filter`
2. `extract.open_filter_setup`
3. `extract.calculate_filters` to initialise the filtering window
4. `extract.open_decay_library`
5. `extract.select_library_decay`
6. `extract.import_library_decays`
7. Select the imported row and run `extract.calculate_filters` again
8. `extract.accept_filters`
9. `extract.select_filter`
10. `extract.apply_all`

The filter calculation checks that a decay component or automatic background range is selected. Import also preserves the GUI compatibility check between the current PTU and library decay TCSPC channel counts. `extract.apply_all` refuses to start when statistical filtering is enabled, but no foreground filter has been selected for an active channel.

## Callback-parity verification

Tests use instrumented Dear PyGui and mode objects to verify ordering at the interaction boundary: widget values are written and the same callback methods used by GUI controls are invoked. Covered paths include initial-button mode transition, decay gating, filter-window initialization, automatic background selection, decay-library import, foreground-filter selection, calibration, automatic ROI controls and Phot2conc calculation.

The repository defines 58 pytest test functions covering the protocol, registry, dispatcher, TCP client/server, panel, callback parity and PTU reader/router. The installed `--test` runner additionally executes four readable numerical scripts for frame reconstruction, concentration formulas, ROI masks and weighted lifetime calculations. Native **Apply to all** and **Calculate all** callbacks retain GUI progress reporting. Error responses identify the failed public command in both the message and `details.command`.


Numerical compatibility is evaluated through explicit expected-versus-calculated values and documented tolerances. Long workflows also report completed, skipped and reused batches so interrupted work can be resumed without repeating accepted results. These checks complement manual GUI parity; representative measurements are still required when adding a new PTU hardware backend or scanner type.